<a href="https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/04_skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matimo Notebook 04: Skills - Progressive Knowledge for Agents

Skills are instructional documents (`SKILL.md`) that give an agent domain-specific knowledge on demand, following the [Agent Skills specification](https://agentskills.io/specification). Instead of stuffing every guideline into a system prompt, an agent loads only what's relevant to the current task - at the cheapest level of detail that answers the question.

### What you'll learn
- The 3 progressive-disclosure levels: metadata → full content → bundled resources
- Creating a skill at runtime with `matimo_create_skill`
- Discovering skills semantically with `matimo_search_skills` (no keyword match needed)
- Loading only the sections you need with `matimo_get_skill_sections` + `matimo_get_skill_content`
- Validating a skill against the spec with `matimo_validate_skill`
- Building a ready-to-inject system-prompt snippet with `build_relevant_skill_prompt`

### Prerequisites
- Complete Notebook 01 (Quickstart) first
- No API key needed - everything in this notebook runs without an LLM

> **Version note:** `matimo_search_skills`, `matimo_get_skill_sections`, and `matimo_get_skill_content` ship in a `matimo-core` release newer than `0.1.3`. If `pip install matimo` gives you `0.1.3` on PyPI, the cells in **Step 5 onward** will return `Tool '...' not found` until the next release goes out - `matimo_create_skill` / `matimo_list_skills` / `matimo_get_skill` / `matimo_validate_skill` (Steps 1–4) work today regardless.

### Step 1 - Install Matimo

In [ ]:
!pip install -q matimo

### Step 2 - Initialize Matimo

Skills live in a `skills_dir` - any folder containing `<skill-name>/SKILL.md` files. We point Matimo at a fresh temp directory so this notebook doesn't touch your filesystem.

`set_global_matimo_instance(matimo)` matters here: the skills meta-tools (`matimo_create_skill`, `matimo_search_skills`, etc.) are `type: function` tools, and they reach the *owning* Matimo instance through this global rather than through whichever instance `.execute()` happens to be called on. Set it once, right after `init()`, before calling any skills meta-tool.

In [1]:
import tempfile
from pathlib import Path

from matimo import Matimo, set_global_matimo_instance

tmp_dir = tempfile.mkdtemp(prefix="matimo-skills-demo-")
skills_dir = Path(tmp_dir) / "skills"
skills_dir.mkdir(parents=True)

matimo = await Matimo.init(auto_discover=True, log_level="silent")
set_global_matimo_instance(matimo)

print(f"Matimo initialized. Skills directory: {skills_dir}")

Matimo initialized. Skills directory: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-skills-demo-as0esemc/skills


### Step 3 - Create a skill with `matimo_create_skill`

A skill is just a directory name plus a `SKILL.md` file with YAML frontmatter (`name`, `description` required). We'll create two: a code-review checklist and a security checklist - enough for the semantic search step later to have something to discriminate between.

In [2]:
CODE_REVIEW_SKILL = """---
name: code-review
description: Checklist and best practices for reviewing pull requests -- correctness, readability, test coverage, and security.
---

# Code Review Checklist

Use this checklist when reviewing a pull request or diff.

## Correctness

- Does the change do what the PR description claims?
- Are edge cases (empty input, null, zero, negative numbers) handled?
- Are there off-by-one errors in loops or slicing?

## Readability

- Are names descriptive enough that comments aren't needed to explain *what* code does?
- Is the diff focused on one concern, or does it bundle unrelated changes?

## Test Coverage

- Does the PR add tests for new behavior?
- Do existing tests still cover the changed code paths?

## Security

- Is user input validated before use?
- Are secrets or credentials hardcoded anywhere in the diff?
"""

SECURITY_CHECKLIST_SKILL = """---
name: security-checklist
description: Security vulnerability detection checklist covering OWASP Top 10 categories and secret handling.
---

# Security Vulnerability Detection Checklist

Run through this checklist before approving code that touches user input, auth, or external calls.

## 1. OWASP Top 10

- **Injection**: Ensure the application is protected against SQL injection, command injection, and OS command injection.
- **Broken Authentication**: Check session handling and credential storage.
- **Sensitive Data Exposure**: Confirm secrets are never logged or printed.

## 2. Secrets Handling

- No hardcoded API keys, passwords, or tokens in source.
- Secrets are read from environment variables or a secrets manager.

## 3. Network

- Validate and allowlist URLs before making outbound requests (SSRF).
- Enforce HTTPS for any external call carrying credentials.
"""

for name, content in [("code-review", CODE_REVIEW_SKILL), ("security-checklist", SECURITY_CHECKLIST_SKILL)]:
    result = await matimo.execute(
        "matimo_create_skill",
        {"name": name, "content": content, "target_dir": str(skills_dir)},
    )
    print(f"{name}: success={result['success']}")
    print(f"  -> {result['path']}")

code-review: success=True
  -> /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-skills-demo-as0esemc/skills/code-review/SKILL.md
security-checklist: success=True
  -> /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-skills-demo-as0esemc/skills/security-checklist/SKILL.md


### Step 4 - Register the skills (Level 1: metadata)

Writing `SKILL.md` to disk doesn't put a skill in the registry by itself - `add_skill_path()` + `reload_skills()` walks the directory and loads it. `get_skills_metadata()` then gives you the cheap, always-safe-to-load view: just names and descriptions. This is what an agent would see at startup, before deciding which skill (if any) is relevant.

In [3]:
from matimo import get_skills_metadata

matimo.add_skill_path(str(skills_dir))
reload_stats = await matimo.reload_skills()
print(f"reload_skills() -> {reload_stats}\n")

for skill in get_skills_metadata(matimo):
    print(f"{skill['name']:<20} {skill['description']}")

reload_skills() -> {'loaded': 2, 'removed': 0}

security-checklist   Security vulnerability detection checklist covering OWASP Top 10 categories and secret handling.
code-review          Checklist and best practices for reviewing pull requests -- correctness, readability, test coverage, and security.


### Step 5 - Semantic search (Level 2 discovery): `matimo_search_skills`

Rather than an agent guessing a skill's exact name, it can search by natural-language intent. This ranks skills by meaning (TF-IDF by default) - the query below doesn't contain the word "security" verbatim in the skill's *name*, only in its description and body.

In [4]:
query = "security vulnerability detection"

search_result = await matimo.execute(
    "matimo_search_skills",
    {"query": query, "limit": 5, "min_score": 0.1},
)

print(f"success={search_result['success']}  total={search_result['total']}\n")
for r in search_result["results"]:
    print(f"  {r['name']:<20} score={r['relevanceScore']:.4f}  {r['description'][:60]}")

top_skill = search_result["results"][0]["name"] if search_result["results"] else None
print(f"\nTop match: {top_skill}")

success=True  total=1

  security-checklist   score=0.5231  Security vulnerability detection checklist covering OWASP To

Top match: security-checklist


### Step 6 - Inventory sections before loading them (Level 2.5): `matimo_get_skill_sections`

Before spending context budget on an entire `SKILL.md`, an agent can ask what sections exist and roughly how many tokens each costs - then request only the ones it needs.

In [5]:
sections_result = await matimo.execute("matimo_get_skill_sections", {"name": top_skill})

print(f"success={sections_result['success']}  total={sections_result['total']}\n")
for s in sections_result["sections"]:
    print(f"  [{s['level']}] {s['path']:<55} ~{s['tokenEstimate']} tokens")

success=True  total=4

  [1] Security Vulnerability Detection Checklist              ~26 tokens
  [2] Security Vulnerability Detection Checklist.1. OWASP Top 10 ~54 tokens
  [2] Security Vulnerability Detection Checklist.2. Secrets Handling ~32 tokens
  [2] Security Vulnerability Detection Checklist.3. Network   ~29 tokens


### Step 7 - Load only what you need, token-budgeted: `matimo_get_skill_content`

`max_tokens` truncates once the budget is hit - useful when you only have a small slice of context to spend on background knowledge. You can also pass `sections=[...]` to load specific headings instead of the whole document.

In [6]:
content_result = await matimo.execute(
    "matimo_get_skill_content",
    {"name": top_skill, "max_tokens": 120},
)

print(f"success={content_result['success']}  tokensUsed={content_result['tokensUsed']}\n")
print(content_result["content"])

success=True  tokensUsed=115

# Security Vulnerability Detection Checklist

Run through this checklist before approving code that touches user input, auth, or external calls.

## 1. OWASP Top 10

- **Injection**: Ensure the application is protected against SQL injection, command injection, and OS command injection.
- **Broken Authentication**: Check session handling and credential storage.
- **Sensitive Data Exposure**: Confirm secrets are never logged or printed.

## 2. Secrets Handling

- No hardcoded API keys, passwords, or tokens in source.
- Secrets are read from environment variables or a secrets manager.


### Step 8 - Validate a skill against the spec: `matimo_validate_skill`

Checks frontmatter, naming rules, and directory structure against the [Agent Skills specification](https://agentskills.io/specification). Useful right after `matimo_create_skill` - including when the *content* came from an LLM rather than a hand-written string, since agent-generated frontmatter can drift from spec.

In [7]:
validate_result = await matimo.execute(
    "matimo_validate_skill", {"name": top_skill, "skills_dir": str(skills_dir)}
)

print(f"valid={validate_result['valid']}")
print(f"issues={validate_result['issues']}")

valid=True
issues=[]


### Step 9 - Build a ready-to-inject system-prompt snippet: `build_relevant_skill_prompt`

This is the piece you'd actually wire into your own agent loop: run it once per request with the user's query, and prepend the result to your system prompt before calling your LLM. It combines Steps 5–7 into one call - semantic search, then full content for the top-K matches only.

In [8]:
from matimo import build_relevant_skill_prompt

prompt = await build_relevant_skill_prompt(
    matimo,
    query,
    top_k=2,
    min_score=0.1,
    header="Apply these skill guidelines:",
)

print(f"{len(prompt)} chars generated\n")
print(prompt[:500] + ("..." if len(prompt) > 500 else ""))

912 chars generated

Apply these skill guidelines:

## Skill: security-checklist (relevance: 0.52)
_Security vulnerability detection checklist covering OWASP Top 10 categories and secret handling._

# Security Vulnerability Detection Checklist

Run through this checklist before approving code that touches user input, auth, or external calls.

## 1. OWASP Top 10

- **Injection**: Ensure the application is protected against SQL injection, command injection, and OS command injection.
- **Broken Authentication**: Check ...


---
## Summary

| Level | Tool / function | What it costs |
|-------|-----------------|----------------|
| 1 - Metadata | `matimo_list_skills` / `get_skills_metadata()` | Name + description for every skill, always cheap |
| 2 - Discovery | `matimo_search_skills` | Ranks skills by meaning, not exact match |
| 2.5 - Inventory | `matimo_get_skill_sections` | Section headings + token estimates, no content loaded |
| 2 - Content | `matimo_get_skill_content` / `matimo_get_skill` | Full or token-budgeted skill body |
| - | `matimo_validate_skill` | Spec-compliance check |
| - | `build_relevant_skill_prompt` | One call combining search + content, ready for a system prompt |

### What's Next?
- `05_mcp_server.ipynb` - expose these same tools over the Model Context Protocol (stdio + HTTP)
- `03_meta_tools.ipynb` - agents that create their own *tools* (not just skills) at runtime
- GitHub: [tallclub/matimo](https://github.com/tallclub/matimo)

**If this was useful, please star the repo: https://github.com/tallclub/matimo**